# Contrôle qualité des numéros de téléphone — EGE FINESS+

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.chargement import resoudre_tables, charger_telecoms, charger_ege, charger_pm
from src.tel_checker import analyser_telephones, marquer_doublons

pd.set_option("display.max_colwidth", 80)
print("Imports OK")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Chargement des EGE FINESS+

Le téléphone n'est plus une colonne de la structure : il vient de la table
`Telecom`, atteinte par la liaison contact et pivotée par canal.

In [2]:
TABLES = resoudre_tables(conn)

print("\nTelecoms :")
telecoms = charger_telecoms(conn, TABLES)

print("\nEGE :")
df_src = charger_ege(conn, TABLES, telecoms)
print(f"\nEGE FINESS actifs chargés : {len(df_src):,}")

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   telecom      -> T_FAC_Telecom
   pm_adresse   -> T_FAC_PmSmsseAdresse
   ege_adresse  -> T_FAC_EgeAdresse
   pm_contact   -> T_FAC_PmSmsseContact
   ege_contact  -> T_FAC_EgeContact

Telecoms :
   3 lignes de canal sentinelle ecartees
   324 doublons (contact, canal) : premiere valeur conservee
   telecoms pivotes : 355,904 contacts, colonnes ['TEL_MAIL', 'TEL_TELECOPIE', 'TEL_TELEPHONE']

EGE :
   228,793 lignes brutes
   228,793 lignes -> 104,667 structures apres deduplication

EGE FINESS actifs chargés : 104,667


## 3. Vue d'ensemble

In [3]:
total     = len(df_src)
renseigne = df_src["TEL_TELEPHONE"].apply(
    lambda x: str(x).strip() not in ("", "nan", "None", "<NA>")).sum()
absent    = total - renseigne

print("─" * 52)
print("PANORAMA DES TÉLÉPHONES EGE FINESS")
print("─" * 52)
print(f"  EGE actifs total            : {total:>10,}")
print(f"  Téléphone renseigné        : {renseigne:>10,}  ({renseigne/total*100:.1f} %)")
print(f"  Téléphone absent           : {absent:>10,}  ({absent/total*100:.1f} %)")
print("─" * 52)

────────────────────────────────────────────────────
PANORAMA DES TÉLÉPHONES EGE FINESS
────────────────────────────────────────────────────
  EGE actifs total            :    104,667
  Téléphone renseigné        :     88,474  (84.5 %)
  Téléphone absent           :     16,193  (15.5 %)
────────────────────────────────────────────────────


## 4. Nettoyage, classification et doublons

In [4]:
print("Nettoyage + classification + doublons...")
df = analyser_telephones(df_src, col_tel="TEL_TELEPHONE", col_commune="TX_CogCommune")
df = marquer_doublons(df, col_tel_clean="telephone_clean", col_id="NB_EgeId")
print(f"Analyse terminée — {len(df):,} EGE traités.")
df[["TX_NomEgeLong", "TEL_TELEPHONE", "telephone_clean", "categorie",
    "motif", "zone_numero", "zone_attendue"]].head(15)

Nettoyage + classification + doublons...
Analyse terminée — 104,667 EGE traités.


,TX_NomEgeLong,TEL_TELEPHONE,telephone_clean,categorie,motif,zone_numero,zone_attendue
0,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,0474454647,0474454647,DOUBLON,numéro partagé par 3 structures distinctes — vérifier si intentionnel,04,04
1,CENTRE HOSPITALIER BUGEY SUD,0479425959,0479425959,VALIDE,,04,04
2,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,0474000989,0474000989,VALIDE,,04,04
3,CENTRE HOSPITALIER DU PAYS DE GEX,0450403838,0450403838,DOUBLON,numéro partagé par 4 structures distinctes — vérifier si intentionnel,04,04
4,CENTRE HOSPITALIER DE MEXIMIEUX,0676721502,0676721502,DOUBLON,numéro partagé par 2 structures distinctes — vérifier si intentionnel,06,04
5,CENTRE HOSPITALIER DE PONT DE VAUX,0385308000,0385308000,DOUBLON,numéro partagé par 2 structures distinctes — vérifier si intentionnel,03,04
6,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,0385362350,0385362350,DOUBLON,numéro partagé par 3 structures distinctes — vérifier si intentionnel,03,04
7,CH INTERCOMMUNAL AIN VAL DE SAONE - THOISSEY,0474049900,0474049900,DOUBLON,numéro partagé par 2 structures distinctes — vérifier si intentionnel,04,04
8,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ESPERANCE,0474408250,0474408250,VALIDE,,04,04
9,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE INTERDEPT,0474408165,0474408165,VALIDE,,04,04


## 5. Distribution des catégories

In [5]:
ordre = ["MANQUANT", "ANOMALIE_CRITIQUE", "SURTAXE", "MOBILE",
         "INCOHERENCE_GEO", "DOUBLON", "VALIDE"]
labels = {
    "MANQUANT": "Manquant (absent / faux vide)",
    "ANOMALIE_CRITIQUE": "[1] Anomalie critique",
    "SURTAXE": "[2] Surtaxe (08 payant)",
    "MOBILE": "[2] Mobile (06/07)",
    "INCOHERENCE_GEO": "[2] Incohérence géographique",
    "DOUBLON": "[2] Doublon",
    "VALIDE": "[0] Valide",
}
vc = df["categorie"].value_counts()

print("─" * 60)
print("DISTRIBUTION DES CATÉGORIES")
print("─" * 60)
for cat in ordre:
    n = int(vc.get(cat, 0))
    print(f"  {labels[cat]:<34} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 60)

────────────────────────────────────────────────────────────
DISTRIBUTION DES CATÉGORIES
────────────────────────────────────────────────────────────
  Manquant (absent / faux vide)      :   16,197  ( 15.5 %)
  [1] Anomalie critique              :       17  (  0.0 %)
  [2] Surtaxe (08 payant)            :       79  (  0.1 %)
  [2] Mobile (06/07)                 :    2,592  (  2.5 %)
  [2] Incohérence géographique       :      109  (  0.1 %)
  [2] Doublon                        :   22,768  ( 21.8 %)
  [0] Valide                         :   62,905  ( 60.1 %)
────────────────────────────────────────────────────────────


## 6. Anomalies critiques — détail par motif

In [6]:
df_crit = df[df["categorie"] == "ANOMALIE_CRITIQUE"].copy()
print(f"Total anomalies critiques : {len(df_crit):,}\n")

COLS = ["TX_NumFinessEge", "TX_NomEgeLong", "TEL_TELEPHONE", "telephone_clean", "motif"]
cols = [c for c in COLS if c in df_crit.columns]

for motif, groupe in df_crit.groupby("motif"):
    print(f"\n{'─'*60}")
    print(f"  {motif}  ({len(groupe):,} cas)")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

Total anomalies critiques : 17


────────────────────────────────────────────────────────────
  longueur 11 != 10  (8 cas)
────────────────────────────────────────────────────────────
TX_NumFinessEge                   TX_NomEgeLong TEL_TELEPHONE telephone_clean             motif
      120002340          PHARMACIE SARAH CHEKIR   00565443110     00565443110 longueur 11 != 10
      220010912              PHARMACIE  PERETTI   02996208134     02996208134 longueur 11 != 10
      270021595            PHARMACIE DU PLATEAU   00232373167     00232373167 longueur 11 != 10
      440016756 PHARMACIE DE SAINT MARC SUR MER   02409109053     02409109053 longueur 11 != 10
      600006282                  PHARMACIE AGEL   00344021582     00344021582 longueur 11 != 10

────────────────────────────────────────────────────────────
  longueur 20 != 10  (1 cas)
────────────────────────────────────────────────────────────
TX_NumFinessEge               TX_NomEgeLong         TEL_TELEPHONE      telephone_clean  

## 7. Signaux qualité — surtaxes, mobiles, incohérences géo, doublons

In [7]:
for cat, titre in [("SURTAXE", "SURTAXES — 08 payant (signal fort)"),
                   ("MOBILE", "MOBILES — 06/07 (signal faible)"),
                   ("INCOHERENCE_GEO", "INCOHÉRENCES GÉO (informatif — portabilité depuis 2023)"),
                   ("DOUBLON", "DOUBLONS (informatif — standard mutualisé possible)")]:
    g = df[df["categorie"] == cat]
    print(f"\n{'─'*60}")
    print(f"  {titre}  ({len(g):,} cas)")
    print(f"{'─'*60}")
    if len(g):
        print(g[cols].head(5).to_string(index=False))


────────────────────────────────────────────────────────────
  SURTAXES — 08 payant (signal fort)  (79 cas)
────────────────────────────────────────────────────────────
TX_NumFinessEge                                          TX_NomEgeLong TEL_TELEPHONE telephone_clean           motif
      040782021 CENTRE DE SOINS DE SUITE ET DE READAPTATION LE COUSSON    0826464677      0826464677 08 payant (082)
      060030335                                        SLD LE MERIDIEN    0826207980      0826207980 08 payant (082)
      060780277                               CLINIQUE SAINTE BRIGITTE    0826960230      0826960230 08 payant (082)
      060780590                                     CLINIQUE DU PALAIS    0825101006      0825101006 08 payant (082)
      060780723                              CLINIQUE DU PARC IMPERIAL    0826307576      0826307576 08 payant (082)

────────────────────────────────────────────────────────────
  MOBILES — 06/07 (signal faible)  (2,592 cas)
───────────────────

## 8. Synthèse finale

In [8]:
n_total = len(df)
compte = df["categorie"].value_counts()

print("=" * 60)
print("SYNTHÈSE GLOBALE — TÉLÉPHONES EGE FINESS")
print("=" * 60)
print(f"  EGE actifs analysés        : {n_total:>8,}")
for cat in ordre:
    print(f"  {labels[cat]:<34} : {int(compte.get(cat, 0)):>8,}")
print("=" * 60)

SYNTHÈSE GLOBALE — TÉLÉPHONES EGE FINESS
  EGE actifs analysés        :  104,667
  Manquant (absent / faux vide)      :   16,197
  [1] Anomalie critique              :       17
  [2] Surtaxe (08 payant)            :       79
  [2] Mobile (06/07)                 :    2,592
  [2] Incohérence géographique       :      109
  [2] Doublon                        :   22,768
  [0] Valide                         :   62,905


## 9. Export Excel — Rapport de signalement

In [9]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/tel")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_tel_ege.xlsx"

COLS_EXPORT = [
    "NB_EgeId", "TX_NumFinessEge", "TX_NomEgeLong",
    "TX_CogCommune", "departement", "adresse_complete",
    "TEL_TELEPHONE", "telephone_clean",
    "categorie", "motif", "zone_numero", "zone_attendue",
]
cols_dispo = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

df_synth = pd.DataFrame([
    {"Catégorie": labels[cat], "Nombre": int(compte.get(cat, 0)),
      "Part": f"{compte.get(cat, 0)/n_total*100:.1f} %"}
    for cat in ordre
] + [{"Catégorie": "TOTAL", "Nombre": n_total, "Part": "100 %"}])

FEUILLES = [
    ("Manquants",         "MANQUANT",          "757575", "EEEEEE"),
    ("Anomalies_Critiques", "ANOMALIE_CRITIQUE", "C62828", "F8CECC"),
    ("Surtaxes",          "SURTAXE",           "C62828", "F8CECC"),
    ("Mobiles",           "MOBILE",            "F57C00", "FFF2CC"),
    ("Incoherences_Geo",  "INCOHERENCE_GEO",   "F57C00", "FFF2CC"),
    ("Doublons",          "DOUBLON",           "F57C00", "FFF2CC"),
    ("Numeros_Valides",   "VALIDE",            "2E7D32", "EBF5EB"),
]

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom_f, categorie, ent, fond in FEUILLES:
        donnees = df[df["categorie"] == categorie][cols_dispo]
        if len(donnees) == 0:
            continue
        donnees.to_excel(writer, sheet_name=nom_f, index=False)
        style_entete(writer.sheets[nom_f], ent)
        remplir(writer.sheets[nom_f], fond)
        auto_width(writer.sheets[nom_f])

print(f"Export OK → {FICHIER_SORTIE.resolve()}")
print("\nFeuilles produites :")
for nom_f, categorie, _, _ in FEUILLES:
    print(f"  {nom_f:<22} : {int(compte.get(categorie, 0)):,}")

Export OK → /home/jovyan/work/finess+_refonte/signalement_data_contact_finess+/results/tel/signalement_tel_ege.xlsx

Feuilles produites :
  Manquants              : 16,197
  Anomalies_Critiques    : 17
  Surtaxes               : 79
  Mobiles                : 2,592
  Incoherences_Geo       : 109
  Doublons               : 22,768
  Numeros_Valides        : 62,905
